<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Mapping 3 Million Points with Lonboard

### CP101: Introduction to Urban Data Analytics

**Week 9, Wednesday** &nbsp;·&nbsp; Thematic Mapping - Map Types & Classification Schemes

**Estimated time:** 15-20 minutes &nbsp;·&nbsp; **Not graded, not submitted.**

<hr style="border: 1px solid #fdb515;" />

### How to work


- Run each code cell with `Shift + Enter`, in order, from the top.
- The map is interactive.
  Drag to pan and scroll to zoom.
- The map opens in a panel on the right of the notebook.
  Later cells change its colors and sizes in place, so watch that panel as you run them.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

By the end of this notebook, you will be able to:

1. Draw more than 3 million points on an interactive map with Lonboard's `ScatterplotLayer`
2. Rescale a variable between two bounds you choose, and explain which points get the end colors
3. Color and size every point by its value with `apply_continuous_cmap()` and the layer's `get_` accessors

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Data for this notebook

**How fast was mobile internet around the world in early 2019?**

[Speedtest by Ookla](https://www.speedtest.net/) is an app that measures the speed of an internet connection.
Ookla averages the results over a grid of square tiles, about 610 meters on a side at the equator and smaller toward the poles, and publishes them as [open data on AWS](https://registry.opendata.aws/speedtest-global-performance/).
Each file covers three months.

This notebook uses the **mobile** (cellular) tiles for January through March 2019, all 3,231,245 of them.
Each row is one tile.

| Column | Meaning |
|---|---|
| `avg_d_kbps` | Average download speed of all tests in the tile, in kilobits per second. 16,000 kbps is 16 megabits per second (Mbps). |
| `geometry` | The center point of the tile |

**Source.** Speedtest® by Ookla® Global Fixed and Mobile Network Performance Maps was accessed on 21 October 2026 from [AWS](https://aws.amazon.com/marketplace/pp/prodview-breawk6ljkovm#resources).
Based on CP101's analysis of Speedtest® by Ookla® Global Fixed and Mobile Network Performance Maps for Q1 2019.
Ookla trademarks used under license and reprinted with permission.
The data is licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).

**Credit.** This notebook is adapted from the [Speedtest tutorial](https://developmentseed.org/lonboard/latest/examples/internet-speeds/) on the Lonboard website.
[Lonboard](https://github.com/developmentseed/lonboard) is built by [Development Seed](https://developmentseed.org/) and released under the MIT License.

## Let's add a new feature to view our maps

[`Sidecar`](https://github.com/jupyter-widgets/jupyterlab-sidecar) opens a panel to the right of the notebook in JupyterLab, so the map stays in view while you scroll and run the cells below.
The next two cells install and import it, and `display()` is what puts the map inside that panel.

In [ ]:
!pip install sidecar -q

In [ ]:
from sidecar import Sidecar
from IPython.display import display
import ipywidgets as widgets

In [ ]:
from pathlib import Path

import geopandas as gpd
import lonboard
import pandas as pd
import shapely
from lonboard import Map, ScatterplotLayer
from palettable.scientific.diverging import Vik_10_r

### Load the tiles

This is the address of the full file for the quarter, about 150 MB.

In [ ]:
url = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2019/quarter=1/2019-01-01_performance_mobile_tiles.parquet"

The next cell first looks for a saved copy at `data/raw/internet-speeds.parquet`.
That copy is already in your folder, so the cell loads it in a few seconds.

Without the saved copy, the cell would download only two columns of the full file, `avg_d_kbps` and `tile`, which can take a while.
The `tile` column stores each square as text, so `shapely.from_wkt()` turns that text into shapes and `shapely.centroid()` keeps each square's center point.
The cell then saves the result, so the download happens only once.

In [ ]:
local_path = Path("data/raw/internet-speeds.parquet")

if local_path.exists():
    gdf = gpd.read_parquet(local_path)
else:
    # download two columns, then keep the center point of each tile
    df = pd.read_parquet(
        url,
        columns=["avg_d_kbps", "tile"]
        )
    tile_geometries = shapely.from_wkt(df["tile"])
    tile_centroids = shapely.centroid(tile_geometries)
    gdf = gpd.GeoDataFrame(
        df[["avg_d_kbps"]],
        geometry=tile_centroids,
        crs="EPSG:4326"
        )
    gdf.to_parquet(local_path)

In [ ]:
gdf.head()

Every row is one tile, and there are more than 3 million of them.

In [ ]:
len(gdf)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Drawing 3 million points

A `ScatterplotLayer` draws one circle for each row.
Lonboard sends the data to your browser in a compact binary format, and your computer's graphics card draws every point at once.
That is how it handles millions of points, far more than `.explore()` can.
`Map()` puts the layer on a basemap.

If you close the side panel by mistake, run the map cell again.
It closes any old panel first, so you only ever have one, but the points start black again, so rerun the cells below it too.

In [ ]:
layer = ScatterplotLayer.from_geopandas(gdf)

m = Map(layer)

m.height = "100vh"
# close the panel from an earlier run, so you keep only one
if "sc" in globals():
    sc.close()

# Display it beside the notebook.
sc = Sidecar(title="Spatial Analysis", anchor="split-right")

with sc:
    display(m)

Every point is black, the default color.
The [documentation for `ScatterplotLayer`](https://developmentseed.org/lonboard/latest/api/layers/scatterplot-layer/) lists the other options it accepts.
`get_fill_color` takes four numbers between 0 and 255, for red, green, blue and opacity, so `[0, 0, 200, 200]` is a slightly see-through blue.
Run the next cell and watch the map in the side panel.

In [ ]:
layer.get_fill_color = [0, 0, 200, 200]

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Coloring points by download speed

Blue is pretty, but the map says more if each point's color shows its download speed.

A colormap expects values between 0 and 1, so first rescale the speeds.
Choose two bounds, `min_bound = 5000` (5 Mbps) and `max_bound = 50000` (50 Mbps).
A tile at `min_bound` becomes 0, a tile at `max_bound` becomes 1, and a tile halfway between them, 27.5 Mbps, becomes 0.5.

In [ ]:
min_bound = 5000
max_bound = 50000
download_speed = gdf["avg_d_kbps"]
normalized_download_speed = (download_speed - min_bound) / (max_bound - min_bound)

Tiles slower than `min_bound` get values below 0, and tiles faster than `max_bound` get values above 1.
The colormap gives all of them its end colors.
That makes choosing the bounds a classification choice, because it decides which tiles share the extreme colors.
Here, 19% of tiles are slower than 5 Mbps and 13.5% are faster than 50 Mbps.

In [ ]:
normalized_download_speed

[`palettable`](https://github.com/jiffyclub/palettable) has many ready-made colormaps.
`Vik_10_r` is a **diverging** colormap, with two dark ends and a pale middle.
Slow tiles are dark orange-brown, middling ones are pale, and fast ones are dark blue.
The `_r` reverses the colormap, so that orange means slow.

In [ ]:
Vik_10_r.mpl_colormap

`apply_continuous_cmap()` looks up each rescaled speed on the colormap and returns one color per point.
`alpha=0.7` makes the points slightly see-through, so overlapping points blend.
Setting `layer.get_fill_color` recolors the map you already drew.

In [ ]:
colors = (
    lonboard
    .colormap
    .apply_continuous_cmap(
        normalized_download_speed,
        Vik_10_r,
        alpha=0.7
        )
)
layer.get_fill_color = colors

Each point in the side panel now has its own color.

### Sizing points by download speed

Any layer property whose name starts with `get_` is an **accessor**.
It takes either one value for every point, like the blue above, or an array with one value per point.

Size the points by download speed too.
`.clip(0, 1)` caps the rescaled speeds at 0 and 1, so no radius is negative, and multiplying by 200 gives a radius of up to 200 meters.
`radius_min_pixels` keeps every point at least half a pixel wide, so the slowest tiles do not vanish when you zoom out.

In [ ]:
layer.get_radius = normalized_download_speed.clip(0, 1) * 200
layer.radius_units = "meters"
layer.radius_min_pixels = 0.5

In the side panel, faster tiles now have bigger circles.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary - Mapping 3 Million Points with Lonboard

| Step | What it does |
|---|---|
| `gpd.read_parquet()` | Loads 3,231,245 tile center points with their download speeds |
| `ScatterplotLayer.from_geopandas()` + `Map()` | Draws one circle per row, on your graphics card |
| `layer.get_fill_color = [0, 0, 200, 200]` | Gives every point the same color |
| `(value - min_bound) / (max_bound - min_bound)` | Rescales the speeds so that 5 Mbps is 0 and 50 Mbps is 1 |
| `lonboard.colormap.apply_continuous_cmap()` | Turns each rescaled speed into a color on a colormap |
| `layer.get_radius` | Sizes each point by its rescaled speed |

## Key Takeaways

1. **Lonboard draws millions of points** - it sends compact binary data to the graphics card, so all 3.2 million tiles fit on one interactive map.
2. **A colormap needs values between 0 and 1** - rescaling between two bounds puts them there.
3. **Bounds are a classification choice** - the 19% of tiles below 5 Mbps all share one end color, so moving a bound changes what the map shows.
4. **Accessors take one value or one per point** - `get_fill_color` and `get_radius` accept either, and setting them updates the map you already drew.

## Further reading

- [Lonboard's Speedtest example](https://developmentseed.org/lonboard/latest/examples/internet-speeds/), which this notebook adapts
- [Ookla Open Data on GitHub](https://github.com/teamookla/ookla-open-data), with every column and quarter
- [Lonboard `ScatterplotLayer` documentation](https://developmentseed.org/lonboard/latest/api/layers/scatterplot-layer/)